# Mirror-CFE Validierung auf MNIST — Klassenpaar 7 vs. 9
**Datensatz:** MNIST (28×28, Graustufen)
**Modell:** ResNet-18 (1 Eingabekanal, von Grund auf trainiert — identisch zum FCVE-MNIST-Notebook)
**Methode:** Mirror Counterfactual Explanations (Mirror-CFE) — Spiegelung des Feature-Vektors
an der Entscheidungsgrenze (Position-Funktion, Eq. 1) + L-BFGS-Verfeinerung, KFE-Übergang
k=0→1, dekodiert über den **Mirror-Decoder** (Skip-Decoder-Basis + SSC = SPE + CSP) und
adversariell trainiert mit PatchGAN — **exakt die Architektur aus `mirror-decoder-xray.ipynb` /
`mirrorcfe-xray.ipynb`**, lediglich an MNIST-Auflösung und Graustufen angepasst.

Dieses Notebook übernimmt **denselben ResNet-Klassifikator für 7 vs. 9** wie das
FCVE-MNIST-Notebook und wendet darauf die **Mirror-CFE-Methode mit dem skip-basierten
Mirror-Decoder** an.

## Anpassung an MNIST (Architektur bleibt die des Röntgen-Notebooks)
Bei 28×28-Eingabe sind die vier ResNet-Feature-Ebenen kleiner als bei 224×224:

| Ebene | Röntgen (224²) | MNIST (28²) |
|-------|----------------|-------------|
| f1 (64)  | 56×56 | **7×7** |
| f2 (128) | 28×28 | **4×4** |
| f3 (256) | 14×14 | **2×2** |
| f4 (512) | 7×7   | **1×1** |

Der Mirror-Decoder, die SPE-Blöcke und die SSC werden 1:1 übernommen, nur die räumlichen
Größen (Upsampling-Pfad 1→2→4→7→14→28) und die Kanalzahl (1 statt 3) sind angepasst.
Der **Kanal-Plan ist identisch** zum Röntgen-Decoder (512→256→128→64→32→16).

**Hinweis zur CSP-Maske:** Da f4 bei MNIST nur 1×1 ist, liefert die CAM-basierte CSP-Maske
keine räumliche Lokalisierung mehr (sie degeneriert zu einem globalen Gate pro Sample).
Der eigentliche Nutzen des Skip-Decoders bleibt aber erhalten: die Skip-Features f1/f2/f3
(7×7 / 4×4 / 2×2) liefern die räumlichen Details, die ein reiner GAP→Bild-Decoder (FCVE)
nicht hat.

## 1. Imports & Konfiguration

In [1]:
import os
import time
import struct
import numpy as np
from pathlib import Path
from array import array
from os.path import join
from tqdm import tqdm
from collections import OrderedDict
from functools import partial

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import DataLoader, Dataset, random_split

torch.manual_seed(2024)
np.random.seed(2024)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Gerät:', DEVICE)

# ── Pfade (MNIST-Rohdaten, IDX-Format, wie auf Kaggle bereitgestellt) ─────────
INPUT_PATH = '/kaggle/input/datasets/hojjatk/mnist-dataset'
TRAIN_IMAGES_PATH = join(INPUT_PATH, 'train-images-idx3-ubyte/train-images-idx3-ubyte')
TRAIN_LABELS_PATH = join(INPUT_PATH, 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
TEST_IMAGES_PATH  = join(INPUT_PATH, 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
TEST_LABELS_PATH  = join(INPUT_PATH, 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')
OUT_DIR = '/kaggle/working'

# ── Klassenpaar (Quelle 7, Ziel 9 — wie FCVE-Paper Fig. 3/4) ──────────────────
SOURCE_DIGIT = 7
TARGET_DIGIT = 9
CLASS_NAMES = {0: str(SOURCE_DIGIT), 1: str(TARGET_DIGIT)}   # Klasse 0 = 7, Klasse 1 = 9

IMG_SIZE   = 28
BATCH_SIZE = 64

# Klassifikator-Hyperparameter (identisch zum FCVE-MNIST-Notebook)
CLF_LR     = 1e-3
CLF_EPOCHS = 10

# ── Mirror-Decoder Hyperparameter (wie mirror-decoder-xray.ipynb) ─────────────
DEC_LR      = 2e-4
DISC_LR     = 2e-4
DEC_EPOCHS  = 15           # MNIST braucht mehr Epochen als der vortrainierte XRAY-Fall
ALPHA       = 0.2          # Triangulation-Relaxation (Paper-Default)
RHO_L       = 0.1          # CSP Maskenschwelle untere Grenze
RHO_U       = 0.5          # CSP Maskenschwelle obere Grenze
LBFGS_ITERS = 20           # L-BFGS-Verfeinerungsschritte bei der Inferenz

# Loss-Gewichte (Paper-Default; W_TRI=1 für Graustufen/kleine Bilder statt 2)
W_CLS = 1.0
W_ADV = 1.0
W_REC = 1.0
W_FEA = 1.0
W_TRI = 1.0

print(f'Klassenpaar: {SOURCE_DIGIT} (Klasse 0) vs. {TARGET_DIGIT} (Klasse 1)')


ModuleNotFoundError: No module named 'tqdm'

## 2. MNIST-Loader (IDX-Format)
Liest die rohen MNIST-Dateien direkt ein (kein torchvision-Download nötig — funktioniert
offline auf Kaggle, sofern der MNIST-Datensatz als Input hinzugefügt wurde).

In [ ]:
class MnistDataloader(object):
    """IDX-Format-Loader für MNIST (identisch zur vom Nutzer bereitgestellten Referenz)."""
    def __init__(self, training_images_filepath, training_labels_filepath,
                 test_images_filepath, test_labels_filepath):
        self.training_images_filepath = training_images_filepath
        self.training_labels_filepath = training_labels_filepath
        self.test_images_filepath = test_images_filepath
        self.test_labels_filepath = test_labels_filepath

    def read_images_labels(self, images_filepath, labels_filepath):
        labels = []
        with open(labels_filepath, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            if magic != 2049:
                raise ValueError(f'Magic number mismatch, expected 2049, got {magic}')
            labels = array("B", file.read())

        with open(images_filepath, 'rb') as file:
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            if magic != 2051:
                raise ValueError(f'Magic number mismatch, expected 2051, got {magic}')
            image_data = array("B", file.read())
        images = []
        for i in range(size):
            images.append([0] * rows * cols)
        for i in range(size):
            img = np.array(image_data[i * rows * cols:(i + 1) * rows * cols])
            img = img.reshape(28, 28)
            images[i][:] = img

        return images, labels

    def load_data(self):
        x_train, y_train = self.read_images_labels(self.training_images_filepath,
                                                     self.training_labels_filepath)
        x_test, y_test = self.read_images_labels(self.test_images_filepath,
                                                  self.test_labels_filepath)
        return (x_train, y_train), (x_test, y_test)


mnist_dataloader = MnistDataloader(TRAIN_IMAGES_PATH, TRAIN_LABELS_PATH,
                                    TEST_IMAGES_PATH, TEST_LABELS_PATH)
(x_train_all, y_train_all), (x_test_all, y_test_all) = mnist_dataloader.load_data()

x_train_all = np.array(x_train_all, dtype=np.uint8)
y_train_all = np.array(y_train_all, dtype=np.int64)
x_test_all  = np.array(x_test_all,  dtype=np.uint8)
y_test_all  = np.array(y_test_all,  dtype=np.int64)

print(f'MNIST geladen — Train: {x_train_all.shape}  Test: {x_test_all.shape}')


## 3. Filtern auf das Klassenpaar & DataLoader
Nur Bilder der Quell- und Zielklasse werden behalten und auf {0,1} ummappt
(0 = Quellklasse 7, 1 = Zielklasse 9).

In [ ]:
def filter_and_remap(images, labels, source_digit, target_digit):
    """Behält nur Bilder von source_digit/target_digit, remapped Labels auf 0/1."""
    mask = (labels == source_digit) | (labels == target_digit)
    imgs_f = images[mask]
    lbls_f = labels[mask]
    remapped = np.where(lbls_f == source_digit, 0, 1).astype(np.int64)
    return imgs_f, remapped


x_train_f, y_train_f = filter_and_remap(x_train_all, y_train_all, SOURCE_DIGIT, TARGET_DIGIT)
x_test_f,  y_test_f  = filter_and_remap(x_test_all,  y_test_all,  SOURCE_DIGIT, TARGET_DIGIT)

print(f'Train gefiltert: {x_train_f.shape[0]} Bilder '
      f'(Klasse 0={int((y_train_f==0).sum())}, Klasse 1={int((y_train_f==1).sum())})')
print(f'Test  gefiltert: {x_test_f.shape[0]} Bilder '
      f'(Klasse 0={int((y_test_f==0).sum())}, Klasse 1={int((y_test_f==1).sum())})')


class MnistPairDataset(Dataset):
    """Liefert (Bild, Label, Dateiname) — Dateiname ist ein synthetischer Index-String,
    damit das Interface zu den Röntgen-/Fire-Notebooks identisch bleibt."""
    def __init__(self, images, labels, augment=False):
        self.images  = images
        self.labels  = labels
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].astype(np.float32) / 255.0   # [0,1]
        if self.augment and np.random.rand() < 0.5:
            img = np.fliplr(img).copy()
        img_t = torch.from_numpy(img).unsqueeze(0)           # (1, 28, 28)
        img_t = (img_t - 0.5) / 0.5                          # Normalisierung auf [-1, 1]
        label = int(self.labels[idx])
        fname = f'{idx:06d}.png'
        return img_t, label, fname


full_train_ds = MnistPairDataset(x_train_f, y_train_f, augment=True)
test_ds       = MnistPairDataset(x_test_f,  y_test_f,  augment=False)

# 90/10 Split des Trainingssatzes für Decoder-Validation (Test-Set bleibt unberührt)
n_val   = int(len(full_train_ds) * 0.1)
n_train = len(full_train_ds) - n_val
generator = torch.Generator().manual_seed(42)
train_split, val_split = random_split(full_train_ds, [n_train, n_val], generator=generator)

full_train_ds_eval = MnistPairDataset(x_train_f, y_train_f, augment=False)
val_dataset   = torch.utils.data.Subset(full_train_ds_eval, val_split.indices)
train_dataset = torch.utils.data.Subset(full_train_ds, train_split.indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,       batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_ds)}')

# ── Sichtprüfung ───────────────────────────────────────────────────────────────
imgs_chk, lbls_chk, _ = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i in range(8):
    img_np = (imgs_chk[i, 0].numpy() * 0.5) + 0.5
    axes[i].imshow(img_np, cmap='gray')
    axes[i].set_title(CLASS_NAMES[int(lbls_chk[i])], fontsize=10)
    axes[i].axis('off')
plt.suptitle('Stichprobe aus train_loader')
plt.tight_layout()
plt.show()


## 4. ResNet-18-Klassifikator definieren
Gleiche Architektur wie in den Röntgen-/Fire-Notebooks (1:1 übernommen), aber mit
`in_channels=1` (MNIST ist Graustufen) und von Grund auf trainiert. Die Struktur
`model.encoder.blocks[0..3]` / `model.decoder.decoder` (linearer Kopf) ist identisch —
damit funktionieren die Mirror-Geometrie und der Skip-Decoder unverändert.

In [ ]:
class Conv2dAuto(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.padding = (self.kernel_size[0] // 2, self.kernel_size[1] // 2)

conv3x3 = partial(Conv2dAuto, kernel_size=3, bias=False)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.blocks   = nn.Identity()
        self.shortcut = nn.Identity()
    def forward(self, x):
        residual = self.shortcut(x) if self.should_apply_shortcut else x
        x = self.blocks(x)
        x += residual
        return x
    @property
    def should_apply_shortcut(self):
        return self.in_channels != self.out_channels

class ResNetResidualBlock(ResidualBlock):
    def __init__(self, in_channels, out_channels, expansion=1,
                 downsampling=1, conv=conv3x3, *args, **kwargs):
        super().__init__(in_channels, out_channels)
        self.expansion    = expansion
        self.downsampling = downsampling
        self.conv         = conv
        self.shortcut = (
            nn.Sequential(OrderedDict({
                'conv': nn.Conv2d(self.in_channels, self.expanded_channels,
                                  kernel_size=1, stride=self.downsampling, bias=False),
                'bn':   nn.BatchNorm2d(self.expanded_channels)
            }))
            if self.should_apply_shortcut else None
        )
    @property
    def expanded_channels(self):
        return self.out_channels * self.expansion
    @property
    def should_apply_shortcut(self):
        return self.in_channels != self.expanded_channels

def conv_bn(in_channels, out_channels, conv, *args, **kwargs):
    return nn.Sequential(OrderedDict({
        'conv': conv(in_channels, out_channels, *args, **kwargs),
        'bn':   nn.BatchNorm2d(out_channels)
    }))

class ResNetBasicBlock(ResNetResidualBlock):
    expansion = 1
    def __init__(self, in_channels, out_channels, activation=nn.ReLU, *args, **kwargs):
        super().__init__(in_channels, out_channels, *args, **kwargs)
        self.blocks = nn.Sequential(
            conv_bn(self.in_channels, self.out_channels,
                    conv=self.conv, bias=False, stride=self.downsampling),
            activation(),
            conv_bn(self.out_channels, self.expanded_channels,
                    conv=self.conv, bias=False),
        )

class ResNetLayer(nn.Module):
    def __init__(self, in_channels, out_channels, block=ResNetBasicBlock,
                 n=1, *args, **kwargs):
        super().__init__()
        downsampling = 2 if in_channels != out_channels else 1
        self.blocks = nn.Sequential(
            block(in_channels, out_channels, *args, **kwargs,
                  downsampling=downsampling),
            *[block(out_channels * block.expansion, out_channels,
                    downsampling=1, *args, **kwargs) for _ in range(n - 1)]
        )
    def forward(self, x):
        return self.blocks(x)

class ResNetEncoder(nn.Module):
    def __init__(self, in_channels=3, blocks_sizes=(64, 128, 256, 512),
                 depths=(2, 2, 2, 2), activation=nn.ReLU,
                 block=ResNetBasicBlock, *args, **kwargs):
        super().__init__()
        self.blocks_sizes = blocks_sizes
        self.gate = nn.Sequential(
            nn.Conv2d(in_channels, blocks_sizes[0], kernel_size=7,
                      stride=2, padding=3, bias=False),
            nn.BatchNorm2d(blocks_sizes[0]),
            activation(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        self.in_out_block_sizes = list(zip(blocks_sizes, blocks_sizes[1:]))
        self.blocks = nn.ModuleList([
            ResNetLayer(blocks_sizes[0], blocks_sizes[0], n=depths[0],
                        activation=activation, block=block, *args, **kwargs),
            *[ResNetLayer(in_ch * block.expansion, out_ch, n=n,
                          activation=activation, block=block, *args, **kwargs)
              for (in_ch, out_ch), n in zip(self.in_out_block_sizes, depths[1:])]
        ])
    def forward(self, x):
        x = self.gate(x)
        for block in self.blocks:
            x = block(x)
        return x

class ResNetDecoder(nn.Module):
    def __init__(self, in_features, n_classes):
        super().__init__()
        self.avg     = nn.AdaptiveAvgPool2d((1, 1))
        self.decoder = nn.Linear(in_features, n_classes)
    def forward(self, x):
        x = self.avg(x)
        x = x.view(x.size(0), -1)
        x = self.decoder(x)
        return x

class ResNet(nn.Module):
    def __init__(self, in_channels, n_classes, *args, **kwargs):
        super().__init__()
        self.encoder = ResNetEncoder(in_channels, *args, **kwargs)
        self.decoder = ResNetDecoder(
            self.encoder.blocks[-1].blocks[-1].expanded_channels, n_classes
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

def build_classifier():
    # in_channels=1 (Graustufen) -- einziger struktureller Unterschied zu den RGB-Notebooks
    return ResNet(in_channels=1, n_classes=2,
                  block=ResNetBasicBlock, depths=[2, 2, 2, 2])

classifier = build_classifier().to(DEVICE)
print(f'ResNet-18 (1 Eingabekanal) Parameter: {sum(p.numel() for p in classifier.parameters()):,}')

# Feature-Map-Größen prüfen (für den Skip-Decoder wichtig)
with torch.no_grad():
    dummy = torch.zeros(2, 1, IMG_SIZE, IMG_SIZE).to(DEVICE)
    x = classifier.encoder.gate(dummy)
    for i, blk in enumerate(classifier.encoder.blocks):
        x = blk(x)
        print(f'  f{i+1}: {tuple(x.shape)}')


## 5. Klassifikator-Training

In [ ]:
clf_optimizer = torch.optim.Adam(classifier.parameters(), lr=CLF_LR)
clf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    clf_optimizer, mode='min', factor=0.5, patience=2)
clf_ce_loss = nn.CrossEntropyLoss()

clf_train_losses, clf_val_losses, clf_val_accs = [], [], []
best_val_acc = 0.0

print('Starte Klassifikator-Training...')
print(f'Epochs: {CLF_EPOCHS}  LR: {CLF_LR}  Batch-Size: {BATCH_SIZE}')
print('-' * 60)

for epoch in range(1, CLF_EPOCHS + 1):
    classifier.train()
    total_loss, n_correct, n_total = 0.0, 0, 0
    for images, labels, _ in tqdm(train_loader, desc=f'Ep {epoch:02d}/{CLF_EPOCHS} Train',
                                   leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = classifier(images)
        loss   = clf_ce_loss(logits, labels)

        clf_optimizer.zero_grad()
        loss.backward()
        clf_optimizer.step()

        total_loss += loss.item() * images.size(0)
        n_correct  += (logits.argmax(1) == labels).sum().item()
        n_total    += images.size(0)

    train_loss = total_loss / n_total
    clf_train_losses.append(train_loss)

    classifier.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels, _ in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = classifier(images)
            val_loss_sum += clf_ce_loss(logits, labels).item() * images.size(0)
            val_correct  += (logits.argmax(1) == labels).sum().item()
            val_total    += images.size(0)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct / val_total
    clf_val_losses.append(val_loss); clf_val_accs.append(val_acc)
    clf_scheduler.step(val_loss)
    print(f'Ep {epoch:02d}/{CLF_EPOCHS}  Train Loss={train_loss:.4f}  '
          f'Val Loss={val_loss:.4f} Acc={val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state_dict': classifier.state_dict(), 'val_acc': val_acc,
                    'class_names': CLASS_NAMES, 'source_digit': SOURCE_DIGIT,
                    'target_digit': TARGET_DIGIT}, os.path.join(OUT_DIR, 'mnist_classifier.pth'))

print(f'\nBester Val-Acc: {best_val_acc:.4f} — Klassifikator gespeichert ✓')

classifier.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for images, labels, _ in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        test_correct += (classifier(images).argmax(1) == labels).sum().item()
        test_total   += images.size(0)
print(f'Test-Set Accuracy ({SOURCE_DIGIT} vs {TARGET_DIGIT}): {test_correct/test_total:.4f}')

# Klassifikator für Mirror-Decoder/CFE einfrieren
for param in classifier.parameters():
    param.requires_grad = False
classifier.eval()
print('Klassifikator eingefroren ✓')


## 6. Feature-Extraktion (alle 4 Ebenen) & Mirror-Geometrie
1:1 aus `mirror-decoder-xray.ipynb` übernommen. `extract_all_features` liefert die vier
Skip-Ebenen, `compute_flk` baut die KFE-Feature-Map f^l_k via gleichmäßigem Kanal-Shift
(garantiert GAP(f^l_k)=z_k). Entscheidungsgrenze: Wm=W[1]-W[0], bm=b[1]-b[0].

In [ ]:
def extract_all_features(model, images):
    """Feature Maps aus allen 4 ResNet-Ebenen (MNIST-Größen):
       f1:(B,64,7,7) f2:(B,128,4,4) f3:(B,256,2,2) f4:(B,512,1,1)"""
    feats = {}
    hooks = []
    layer_map = {'f1': model.encoder.blocks[0], 'f2': model.encoder.blocks[1],
                 'f3': model.encoder.blocks[2], 'f4': model.encoder.blocks[3]}
    for name, layer in layer_map.items():
        hooks.append(layer.register_forward_hook(
            lambda m, i, o, n=name: feats.update({n: o})))
    with torch.no_grad():
        _ = model(images)
    for h in hooks:
        h.remove()
    return feats['f1'], feats['f2'], feats['f3'], feats['f4']

def get_boundary_params(model):
    """Mirror-Hyperebene: Wm = W[1]-W[0], bm = b[1]-b[0] (Richtung Klasse 1)."""
    W  = model.decoder.decoder.weight.data   # (2, 512)
    b  = model.decoder.decoder.bias.data     # (2,)
    return (W[1] - W[0]).clone(), (b[1] - b[0]).clone()

def gap(fmap):
    return F.adaptive_avg_pool2d(fmap, (1, 1)).flatten(1)

def classify_from_gap(model, z):
    """Logits aus einem (ggf. modifizierten) GAP-Latent z (B,512)."""
    return model.decoder.decoder(z)

def predict_probs_from_z(model, z):
    """Softmax-Wahrscheinlichkeiten direkt aus einem GAP-Latent z (B,512)."""
    return torch.softmax(model.decoder.decoder(z), dim=1)

def compute_flk(f4_s, zs, Wm, bm, k):
    """Supp. Eq. 16/17: f^l_k = f^l_s + z_Δ (z_Δ broadcast über H,W). Garantiert GAP(f^l_k)=z_k."""
    B, C, H, Wd = f4_s.shape
    W_hat = Wm / (Wm.norm() + 1e-8)                         # (C,)
    dot   = (Wm.view(1, C) * zs).sum(1, keepdim=True) + bm  # (B,1)
    z_delta = -2.0 * k * dot * W_hat.view(1, C)             # (B,C)
    zk  = zs + z_delta                                      # (B,C)
    flk = f4_s + z_delta.view(B, C, 1, 1)                   # broadcast
    return flk, zk

# Grenze global bestimmen + CAM-Gewicht für CSP
Wm, bm = get_boundary_params(classifier)
Wm, bm = Wm.to(DEVICE), bm.to(DEVICE)
Wmat   = classifier.decoder.decoder.weight.data.clone().to(DEVICE)   # (2, 512)

# Sanity Check: GAP(f^l_k) == z_k
with torch.no_grad():
    _imgs, _, _ = next(iter(val_loader)); _imgs = _imgs.to(DEVICE)
    _f1, _f2, _f3, _f4 = extract_all_features(classifier, _imgs)
    print('Feature-Map-Shapes:', _f1.shape, _f2.shape, _f3.shape, _f4.shape)
    _zs = gap(_f4)
    _flk, _zk = compute_flk(_f4, _zs, Wm, bm, k=0.8)
    print(f'Sanity: max|GAP(f^l_k) - z_k| = {(gap(_flk) - _zk).abs().max().item():.2e}  (~0) ✓')


## 7. Skip-Connection-Controller (SSC = SPE + CSP)
1:1 aus `mirror-decoder-xray.ipynb`, nur die räumlichen Größen sind auf MNIST angepasst:
die SPE-Bottlenecks poolen auf die f4-Größe (hier 1×1 statt 7×7), die SPE-Ausgaben und
CSP-Masken auf die jeweilige Skip-Ebene (7×7 / 4×4 / 2×2).

In [ ]:
class SPEBlock(nn.Module):
    """Spatial Pattern Editor für eine Skip-Ebene (Paper Eq. 12).
    flk_hw = räumliche Größe von f^l_k (bei MNIST 1, bei Röntgen 7)."""
    def __init__(self, Ci, Hi, flk_ch=512, flk_hw=1, bottleneck_ch=128):
        super().__init__()
        self.Hi = Hi
        # B_i: Quell-Skip-Feature → (bottleneck_ch, flk_hw, flk_hw)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(Ci, bottleneck_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(bottleneck_ch),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((flk_hw, flk_hw)),
        )
        # D_i: concat(bottleneck, f^l_k) → zurück auf (Ci, Hi, Hi)
        self.decoder = nn.Sequential(
            nn.Conv2d(bottleneck_ch + flk_ch, Ci, kernel_size=3, padding=1),
            nn.BatchNorm2d(Ci),
            nn.ReLU(inplace=True),
            nn.Upsample(size=(Hi, Hi), mode='bilinear', align_corners=False),
            nn.Conv2d(Ci, Ci, kernel_size=3, padding=1),
            nn.BatchNorm2d(Ci),
            nn.ReLU(inplace=True),
        )
    def forward(self, fi_s, flk):
        bi  = self.bottleneck(fi_s)             # (B, bn, flk_hw, flk_hw)
        cat = torch.cat([bi, flk], dim=1)       # (B, bn+512, flk_hw, flk_hw)
        return self.decoder(cat)                # (B, Ci, Hi, Hi)


class SSC(nn.Module):
    """Skip-Connection-Controller = SPE (3 Ebenen) + CSP-Maskierung. MNIST-Größen."""
    def __init__(self, rho_l=0.1, rho_u=0.5):
        super().__init__()
        self.rho_l = rho_l
        self.rho_u = rho_u
        # SPE-Blöcke für f1,f2,f3 (f4 = f^l_k ist Haupteingang, kein SPE nötig)
        self.spe1 = SPEBlock(64,  7, flk_hw=1)
        self.spe2 = SPEBlock(128, 4, flk_hw=1)
        self.spe3 = SPEBlock(256, 2, flk_hw=1)

    def csp_mask(self, flk, W, k, target_hw, src_cls, tgt_cls):
        """Eq. 13: CAM-Union-Maske, hochskaliert auf target_hw.
        Bei MNIST ist flk 1×1 → die Maske ist räumlich uniform (globales Gate pro Sample)."""
        B, C, H, Wd = flk.shape
        Uk = torch.einsum('oc,bchw->bohw', W, flk)         # (B,|C|,H,W)
        Uk = F.relu(Uk)
        mx = Uk.amax(dim=(2, 3), keepdim=True) + 1e-8
        Nk = Uk / mx                                        # (B,|C|,H,W) in [0,1]
        rho = min(max(1.0 - k, self.rho_l), self.rho_u)
        idx = torch.arange(B, device=flk.device)
        Ns = (Nk[idx, src_cls].unsqueeze(1) > rho).float()  # (B,1,H,W)
        Nt = (Nk[idx, tgt_cls].unsqueeze(1) > rho).float()
        M  = torch.clamp(Ns + Nt, 0.0, 1.0)                 # Union
        return F.interpolate(M, size=target_hw, mode='nearest')

    def forward(self, f1_s, f2_s, f3_s, flk, W, k, src_cls, tgt_cls):
        u1 = self.spe1(f1_s, flk)
        u2 = self.spe2(f2_s, flk)
        u3 = self.spe3(f3_s, flk)
        m1 = self.csp_mask(flk, W, k, (7, 7), src_cls, tgt_cls)
        m2 = self.csp_mask(flk, W, k, (4, 4), src_cls, tgt_cls)
        m3 = self.csp_mask(flk, W, k, (2, 2), src_cls, tgt_cls)
        fp1 = (1 - m1) * f1_s + m1 * u1
        fp2 = (1 - m2) * f2_s + m2 * u2
        fp3 = (1 - m3) * f3_s + m3 * u3
        return fp1, fp2, fp3

print('SSC (SPE + CSP) definiert ✓')


## 8. Mirror-Decoder (Skip-Decoder-Basis + SSC) & PatchGAN-Diskriminator
1:1 aus `mirror-decoder-xray.ipynb`. Der Upsampling-Pfad ist auf MNIST angepasst
(1→2→4→7→14→28 statt 7→14→28→56→112→224); der **Kanal-Plan ist identisch**
(512→256→128→64→32→16). Out-Conv liefert 1 Kanal (Graustufen) statt 3.

In [ ]:
class MirrorDecoder(nn.Module):
    """SkipDecoder-Basis (wie FCVE) + integriertes SSC. MNIST-Auflösung."""
    def __init__(self, rho_l=0.1, rho_u=0.5):
        super().__init__()
        self.ssc = SSC(rho_l=rho_l, rho_u=rho_u)
        # ── SkipDecoder-Basis ── (size = Zielgröße der jeweiligen Ebene)
        self.up1 = self._up_block(512, 256, 2)    # 1  → 2
        self.up2 = self._up_block(512, 128, 4)    # 2  → 4   (+skip f'3: 256)
        self.up3 = self._up_block(256, 64,  7)    # 4  → 7   (+skip f'2: 128)
        self.up4 = self._up_block(128, 32,  14)   # 7  → 14  (+skip f'1: 64)
        self.up5 = self._up_block(32,  16,  28)   # 14 → 28
        self.out_conv = nn.Sequential(
            nn.Conv2d(16, 1, kernel_size=3, padding=1),
            nn.Tanh()
        )

    def _up_block(self, in_ch, out_ch, size):
        return nn.Sequential(
            nn.Upsample(size=(size, size), mode='bilinear', align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def _decode(self, flk, fp1, fp2, fp3):
        x = self.up1(flk)                  # (B,256,2,2)
        x = torch.cat([x, fp3], dim=1)     # (B,512,2,2)
        x = self.up2(x)                    # (B,128,4,4)
        x = torch.cat([x, fp2], dim=1)     # (B,256,4,4)
        x = self.up3(x)                    # (B,64,7,7)
        x = torch.cat([x, fp1], dim=1)     # (B,128,7,7)
        x = self.up4(x)                    # (B,32,14,14)
        x = self.up5(x)                    # (B,16,28,28)
        return self.out_conv(x)            # (B,1,28,28)

    def forward(self, flk, f1_s, f2_s, f3_s, W, k, src_cls, tgt_cls):
        """flk: Haupteingang (B,512,1,1); f{1,2,3}_s: Quell-Skip-Features;
           W: Klassifikator-Gewicht (|C|,512) für CAM; k: Skalar-Stepfaktor;
           src_cls/tgt_cls: (B,) Klassenindizes für CSP-Maske."""
        fp1, fp2, fp3 = self.ssc(f1_s, f2_s, f3_s, flk, W, k, src_cls, tgt_cls)
        return self._decode(flk, fp1, fp2, fp3)


class PatchDiscriminator(nn.Module):
    """PatchGAN-Diskriminator für L_adv (Paper Eq. 3). 1-Kanal, 28×28."""
    def __init__(self, in_ch=1, ndf=32):
        super().__init__()
        def block(i, o, bn=True):
            layers = [nn.Conv2d(i, o, 4, stride=2, padding=1)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        self.net = nn.Sequential(
            *block(in_ch, ndf, bn=False),   # 28→14
            *block(ndf, ndf*2),             # 14→7
            *block(ndf*2, ndf*4),           # 7→3
            nn.Conv2d(ndf*4, 1, 3, stride=1, padding=1)   # PatchMap 3×3
        )
    def forward(self, x):
        return self.net(x)


decoder = MirrorDecoder(rho_l=RHO_L, rho_u=RHO_U).to(DEVICE)
disc    = PatchDiscriminator().to(DEVICE)
print(f'MirrorDecoder Parameter: {sum(p.numel() for p in decoder.parameters()):,}')
print(f'Discriminator Parameter: {sum(p.numel() for p in disc.parameters()):,}')

# Forward-Smoke-Test
with torch.no_grad():
    _imgs, _lbls, _ = next(iter(train_loader)); _imgs = _imgs.to(DEVICE)
    _f1, _f2, _f3, _f4 = extract_all_features(classifier, _imgs)
    _zs = gap(_f4)
    _src = classifier(_imgs).argmax(1); _tgt = 1 - _src
    _flk, _zk = compute_flk(_f4, _zs, Wm, bm, k=0.7)
    _out = decoder(_flk, _f1, _f2, _f3, Wmat, 0.7, _src, _tgt)
    print('Decoder Output:', tuple(_out.shape), '| Disc Output:', tuple(disc(_out).shape))


## 9. Loss-Funktionen
Wie `mirror-decoder-xray.ipynb` (Eq. 2/3/5/6–11), für MNIST: 1-Kanal-Normalisierung
(x-0.5)/0.5 statt ImageNet-Mean/Std. `loss_fea` nutzt eine **gradientenfähige**
Feature-Extraktion (der Klassifikator ist eingefroren, Gradienten fließen nur durch die
Aktivierungen zurück zum dekodierten Bild).

In [ ]:
mae = nn.L1Loss()

def tanh_to_img(t):
    """Decoder-Output [-1,1] → [0,1]."""
    return (t + 1.0) / 2.0

def denormalise(t):
    """MNIST [-1,1] → [0,1]."""
    return (t * 0.5 + 0.5).clamp(0, 1)

def renormalise(img01):
    """[0,1] → [-1,1] (Eingabe-Normalisierung des Klassifikators)."""
    return (img01 - 0.5) / 0.5

def gap_f4_grad(model, images):
    """GAP der letzten Feature-Map MIT Gradientenfluss (kein no_grad)."""
    feats = []
    h = model.encoder.blocks[-1].register_forward_hook(lambda m, i, o: feats.append(o))
    _ = model(images)
    h.remove()
    return gap(feats[0])

def loss_cls(xk01, zk):
    """Eq. 2: KLD(σ(F(G(z))), σ(Kopf(z_k))). xk01 in [0,1]."""
    logp_hat = F.log_softmax(classifier(renormalise(xk01)), dim=1)
    p_tgt    = torch.softmax(classify_from_gap(classifier, zk), dim=1).detach()
    return F.kl_div(logp_hat, p_tgt, reduction='batchmean')

def loss_fea(xk01, zk):
    """Eq. 5: ||z_k - GAP(F(G(z_k)))||_2."""
    zk_re = gap_f4_grad(classifier, renormalise(xk01))
    return (zk - zk_re).norm(dim=1).mean()

def loss_tri(xk01, xs01, xref01, zk, zs, zref, alpha=0.2):
    """Eq. 6–11 (ein Zweig): hält |x_k-x_ref|/|x_s-x_k| ≈ ||z_k-z_ref||/||z_s-z_k||.
       Distanzen als L1-Pixel-MITTELWERTE."""
    beta   = (zk - zref).norm(dim=1) / ((zs - zk).norm(dim=1) + 1e-8)
    d_kref = (xk01 - xref01).abs().flatten(1).mean(1)
    d_sk   = (xs01 - xk01).abs().flatten(1).mean(1)
    lower = torch.clamp((1 - alpha) / beta * d_kref - d_sk, min=0)
    upper = torch.clamp(d_sk - (1 + alpha) / beta * d_kref, min=0)
    return (lower + upper).mean()

def adv_g(disc, fake):
    pred = disc(fake)
    return F.binary_cross_entropy_with_logits(pred, torch.ones_like(pred))

def adv_d(disc, real, fake):
    pr = disc(real); pf = disc(fake.detach())
    lr = F.binary_cross_entropy_with_logits(pr, torch.ones_like(pr))
    lf = F.binary_cross_entropy_with_logits(pf, torch.zeros_like(pf))
    return 0.5 * (lr + lf)

print('Loss-Funktionen definiert ✓')


## 10. Mirror-Decoder Training (cls + adv + rec + fea + tri)
1:1 aus `mirror-decoder-xray.ipynb`: pro Batch ein zufälliges k∈[0,1], KFE-Feature
`flk = compute_flk(...)`, Generator-/Diskriminator-Schritt, separater k=0-Forward für
die Rekonstruktion. Referenzbild für L_tri: Zielklasse (k≥0.5) bzw. Quellklasse (k<0.5).

In [ ]:
def pick_reference(images, labels, want_cls):
    """Wählt pro Sample ein echtes Referenzbild der Klasse want_cls aus dem Batch.
       Fallback: irgendein anderes Bild, falls Klasse nicht vorhanden."""
    B = images.size(0)
    idx_out = torch.empty(B, dtype=torch.long)
    for i in range(B):
        cand = (labels == want_cls[i]).nonzero(as_tuple=True)[0]
        cand = cand[cand != i]
        if len(cand) == 0:
            cand = torch.arange(B)[torch.arange(B) != i]
        idx_out[i] = cand[torch.randint(len(cand), (1,))]
    return images[idx_out]


dec_opt   = torch.optim.Adam(decoder.parameters(), lr=DEC_LR,  betas=(0.5, 0.999))
disc_opt  = torch.optim.Adam(disc.parameters(),    lr=DISC_LR, betas=(0.5, 0.999))
dec_sched = torch.optim.lr_scheduler.CosineAnnealingLR(dec_opt, T_max=DEC_EPOCHS)

hist = {'cls':[], 'adv':[], 'rec':[], 'fea':[], 'tri':[], 'total':[], 'val_cls':[]}

print('Starte Mirror-Decoder Training...')
print(f'Epochs: {DEC_EPOCHS}  LR: {DEC_LR}  Batch: {BATCH_SIZE}  α: {ALPHA}  W_TRI: {W_TRI}')
print('-' * 70)

for epoch in range(1, DEC_EPOCHS + 1):
    decoder.train(); disc.train()
    agg = {kk: 0.0 for kk in ['cls','adv','rec','fea','tri','total']}
    n_seen = 0

    for images, labels, _ in tqdm(train_loader, desc=f'Ep {epoch:02d}/{DEC_EPOCHS}', leave=False):
        images = images.to(DEVICE); labels = labels.to(DEVICE)
        B = images.size(0)

        # ── Features + Klassen ──
        f1, f2, f3, f4 = extract_all_features(classifier, images)
        zs  = gap(f4)
        src = classify_from_gap(classifier, zs).argmax(1)
        tgt = 1 - src

        # ── k zufällig aus [0,1] ──
        k = float(torch.rand(1).item())
        flk, zk = compute_flk(f4, zs, Wm, bm, k)

        # ── Referenz-Latents/Bilder für L_tri ──
        ref_cls = tgt if k >= 0.5 else src
        x_ref = pick_reference(images, labels, ref_cls)
        with torch.no_grad():
            _, _, _, f4_ref = extract_all_features(classifier, x_ref)
            z_ref  = gap(f4_ref)
        xref01 = denormalise(x_ref)
        xs01   = denormalise(images)

        # ── Generator-Forward (KFE) ──
        xk   = decoder(flk, f1, f2, f3, Wmat, k, src, tgt)
        xk01 = tanh_to_img(xk)

        # ════ Discriminator-Schritt ════
        disc_opt.zero_grad()
        ld = adv_d(disc, xs01, xk01)
        ld.backward()
        disc_opt.step()

        # ════ Generator-Schritt ════
        dec_opt.zero_grad()
        flk0, _ = compute_flk(f4, zs, Wm, bm, 0.0)               # k=0 → Rekonstruktion
        xrec = decoder(flk0, f1, f2, f3, Wmat, 0.0, src, src)
        l_rec = mae(tanh_to_img(xrec), xs01)

        l_cls = loss_cls(xk01, zk)
        l_fea = loss_fea(xk01, zk)
        l_adv = adv_g(disc, xk01)
        l_tri = loss_tri(xk01, xs01, xref01, zk, zs, z_ref, alpha=ALPHA)

        total = (W_CLS*l_cls + W_ADV*l_adv + W_REC*l_rec + W_FEA*l_fea + W_TRI*l_tri)
        total.backward()
        dec_opt.step()

        for kk, vv in zip(['cls','adv','rec','fea','tri','total'],
                          [l_cls,l_adv,l_rec,l_fea,l_tri,total]):
            agg[kk] += float(vv) * B
        n_seen += B

    dec_sched.step()
    for kk in agg:
        hist[kk].append(agg[kk] / n_seen)

    # ── Validation: L_cls auf k=1 (Reflexion, härtester Fall) ──
    decoder.eval()
    vcls, vn = 0.0, 0
    with torch.no_grad():
        for images, labels, _ in val_loader:
            images = images.to(DEVICE); B = images.size(0)
            f1, f2, f3, f4 = extract_all_features(classifier, images)
            zs = gap(f4); src = classify_from_gap(classifier, zs).argmax(1); tgt = 1 - src
            flk, zk = compute_flk(f4, zs, Wm, bm, 1.0)
            xk = decoder(flk, f1, f2, f3, Wmat, 1.0, src, tgt)
            vcls += float(loss_cls(tanh_to_img(xk), zk)) * B
            vn   += B
    hist['val_cls'].append(vcls / vn)

    print(f'Ep {epoch:02d} | cls {hist["cls"][-1]:.4f}  adv {hist["adv"][-1]:.3f}  '
          f'rec {hist["rec"][-1]:.4f}  fea {hist["fea"][-1]:.3f}  tri {hist["tri"][-1]:.4f}  '
          f'| val_cls(k=1) {hist["val_cls"][-1]:.4f}')

torch.save({'model_state_dict': decoder.state_dict(), 'history': hist, 'dataset': 'mnist',
            'config': {'alpha': ALPHA, 'rho_l': RHO_L, 'rho_u': RHO_U}},
           os.path.join(OUT_DIR, 'mirror_decoder_mnist.pth'))
print('\nTraining abgeschlossen — Decoder gespeichert ✓')


## 11. Trainingskurven

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(hist['total']) + 1)
for kk in ['cls', 'rec', 'fea', 'tri']:
    axes[0].plot(ep, hist[kk], 'o-', ms=3, label=kk)
axes[0].plot(ep, hist['val_cls'], 's--', ms=3, label='val_cls (k=1)')
axes[0].set_title('Mirror-Decoder Losses'); axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(ep, hist['adv'],   'o-', ms=3, color='crimson',  label='adv (Generator)')
axes[1].plot(ep, hist['total'], 'o-', ms=3, color='black',    label='total')
axes[1].set_title('Adversarial / Total'); axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 12. Decoder-Qualitätscheck — Rekonstruktion (k=0) & CFE-Flip (k=1)

In [ ]:
decoder.eval()
imgs, lbls, _ = next(iter(val_loader))
imgs = imgs.to(DEVICE)
n = min(6, imgs.size(0))

with torch.no_grad():
    f1, f2, f3, f4 = extract_all_features(classifier, imgs)
    zs  = gap(f4)
    src = classify_from_gap(classifier, zs).argmax(1); tgt = 1 - src
    flk0, _ = compute_flk(f4, zs, Wm, bm, 0.0)
    flk1, _ = compute_flk(f4, zs, Wm, bm, 1.0)
    rec   = decoder(flk0, f1, f2, f3, Wmat, 0.0, src, src)
    cfe   = decoder(flk1, f1, f2, f3, Wmat, 1.0, src, tgt)
    p_cfe = predict_probs_from_z(classifier, gap(flk1))[:, 1]

fig, axes = plt.subplots(3, n, figsize=(2.2*n, 6.6))
for i in range(n):
    axes[0, i].imshow(denormalise(imgs[i, 0].cpu()).numpy(), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(tanh_to_img(rec[i, 0].cpu()).clamp(0,1).numpy(), cmap='gray'); axes[1, i].axis('off')
    axes[2, i].imshow(tanh_to_img(cfe[i, 0].cpu()).clamp(0,1).numpy(), cmap='gray'); axes[2, i].axis('off')
    flip = (p_cfe[i].item() >= 0.5) == (tgt[i].item() == 1)
    axes[0, i].set_title(f'{CLASS_NAMES[src[i].item()]}', fontsize=9)
    axes[2, i].set_title(f'→{CLASS_NAMES[tgt[i].item()]} ({p_cfe[i].item():.0%})',
                         fontsize=8, color='green' if flip else 'red')
axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Rekon (k=0)', fontsize=10)
axes[2, 0].set_ylabel('CFE (k=1)', fontsize=10)
plt.suptitle('Mirror-Decoder: Rekonstruktion & geometrische Reflexion', fontsize=12)
plt.tight_layout(); plt.show()


## 13. Mirror-CFE Pipeline (Inferenz) + KFE-Visualisierung
`position_function` (Eq. 1) + `refine_with_lbfgs` aus `mirrorcfe-xray.ipynb`. Der
KFE-Übergang interpoliert die Feature-Map linear von f4_s (k=0) zur verfeinerten
Reflexion (k=1) und dekodiert **jeden Schritt mit dem Mirror-Decoder** (inkl. Skips).

In [ ]:
def position_function(zs_batch, Wm, bm, k):
    """Paper Eq. 1: P(zs, Wm, bm, k) = zs - 2k*(Wm^T*zs+bm)*Ŵm  (auf Feature-Map)."""
    B, C, H, W = zs_batch.shape
    W_hat   = Wm / (Wm.norm() + 1e-8)
    zs_flat = zs_batch.view(B, C, -1)
    dot     = (Wm.view(1, C, 1) * zs_flat).sum(dim=1, keepdim=True)
    scalar  = dot + bm.item()
    delta   = 2 * k * scalar * W_hat.view(1, C, 1)
    return (zs_flat - delta).view(B, C, H, W)

def refine_with_lbfgs(model, zr_init, source_labels, cfe_labels,
                      orig_probs_2cls, num_iterations=20):
    """L-BFGS-Verfeinerung, bis der Kopf die Zielklasse (vertauschte Probs) ausgibt."""
    swapped = orig_probs_2cls.clone().detach()
    idx     = torch.arange(len(cfe_labels))
    src     = source_labels.view(-1); cfe = cfe_labels.view(-1)
    tmp               = swapped[idx, cfe].clone()
    swapped[idx, cfe] = swapped[idx, src]
    swapped[idx, src] = tmp

    z         = Variable(zr_init.clone().detach(), requires_grad=True)
    optimizer = torch.optim.LBFGS([z], lr=0.1)
    def closure():
        optimizer.zero_grad()
        flat   = torch.flatten(F.adaptive_avg_pool2d(z, (1, 1)), 1)
        probs  = torch.softmax(model.decoder.decoder(flat), dim=1)
        loss   = torch.norm(probs - swapped) ** 2
        loss.backward()
        return loss
    for _ in range(num_iterations):
        optimizer.step(closure)
    return z

def compute_mirror_cfe(model, images, device, num_iterations=20):
    """Vollständige Mirror-CFE Pipeline. Rückgabe:
       mirror_fv (B,512,1,1), f1,f2,f3 (Quell-Skips), cfe_labels, source_labels, orig_probs."""
    model.eval()
    images = images.to(device)
    with torch.no_grad():
        logits        = model(images)
        probs         = torch.softmax(logits, dim=1)
        orig_probs    = probs[:, 1]
        source_labels = logits.argmax(dim=1)
        cfe_labels    = 1 - source_labels

    f1, f2, f3, f4 = extract_all_features(model, images)
    with torch.no_grad():
        zr_geometric = position_function(f4.clone(), Wm, bm, k=1.0)
    mirror_fv = refine_with_lbfgs(model, zr_geometric, source_labels, cfe_labels,
                                  probs.clone(), num_iterations=num_iterations)
    return mirror_fv, f1, f2, f3, cfe_labels, source_labels, orig_probs

def predict_from_features(model, feature_maps):
    """P(Klasse 1) für eine Feature-Map (B,512,1,1)."""
    with torch.no_grad():
        return torch.softmax(model.decoder.decoder(gap(feature_maps)), dim=1)[:, 1]


def visualise_mirror_cfe(model, decoder, images, labels, n_samples=4,
                         num_iterations=20, save_path='mirror_cfe_mnist.png'):
    model.eval(); decoder.eval()
    images = images.to(DEVICE)
    n = min(n_samples, images.size(0))
    imgs = images[:n]

    mirror_fv, f1, f2, f3, cfe_lbl, src_lbl, orig_p = compute_mirror_cfe(
        model, imgs, DEVICE, num_iterations=num_iterations)
    _, _, _, f4_s = extract_all_features(model, imgs)
    mf = mirror_fv.detach()

    k_steps  = [0.0, 0.25, 0.5, 0.6, 0.75, 1.0]
    k_labels = ['k=0\n(Quelle)', 'k=0.25', 'k=0.5\n(Grenze)',
                'k=0.6', 'k=0.75', 'k=1.0\n(Reflexion)']

    dec_imgs, conf = [], []
    with torch.no_grad():
        for k in k_steps:
            flk_k = (1 - k) * f4_s + k * mf                       # lineare KFE-Interpolation
            img_k = decoder(flk_k, f1, f2, f3, Wmat, k, src_lbl, cfe_lbl)
            dec_imgs.append(tanh_to_img(img_k).clamp(0, 1).cpu())
            conf.append(torch.softmax(model.decoder.decoder(gap(flk_k)), dim=1)[:, 1].cpu().numpy())
        cfe_probs = predict_from_features(model, mf).cpu().numpy()

    orig_np = orig_p.detach().cpu().numpy()
    n_cols  = 2 + len(k_steps)
    fig = plt.figure(figsize=(n_cols * 2.4, n * 3.2))
    gs  = gridspec.GridSpec(n, n_cols, figure=fig, hspace=0.55, wspace=0.25)

    for i in range(n):
        s_lbl, c_lbl, t_lbl = src_lbl[i].item(), cfe_lbl[i].item(), int(labels[i])
        flipped = int(cfe_probs[i] >= 0.5) == c_lbl

        ax0 = fig.add_subplot(gs[i, 0])
        ax0.imshow(denormalise(imgs[i, 0].cpu()).numpy(), cmap='gray')
        p_src = orig_np[i] if s_lbl == 1 else 1 - orig_np[i]
        ax0.set_title(f'Original\nWahr: {CLASS_NAMES[t_lbl]}\nPred: {CLASS_NAMES[s_lbl]} ({p_src:.0%})',
                      fontsize=7, color='limegreen' if s_lbl == t_lbl else 'tomato')
        ax0.axis('off')

        for j, k_lbl in enumerate(k_labels):
            ax = fig.add_subplot(gs[i, j + 1])
            ax.imshow(dec_imgs[j][i, 0].numpy(), cmap='gray', vmin=0, vmax=1)
            p_k = conf[j][i]
            ax.text(0.5, 0.02, f'P(Kl.1)={p_k:.2f}\n{CLASS_NAMES[int(p_k >= 0.5)]}',
                    transform=ax.transAxes, ha='center', va='bottom', fontsize=6,
                    color='cyan' if int(p_k >= 0.5) == c_lbl else 'white',
                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
            if abs(k_steps[j] - 0.5) < 0.01:
                for sp in ax.spines.values(): sp.set_edgecolor('cyan'); sp.set_linewidth(2)
            if k_steps[j] == 1.0:
                fc = 'limegreen' if flipped else 'tomato'
                for sp in ax.spines.values(): sp.set_edgecolor(fc); sp.set_linewidth(2)
            ax.set_title(k_lbl, fontsize=6.5); ax.set_xticks([]); ax.set_yticks([])

        ax_c = fig.add_subplot(gs[i, -1])
        ci = [conf[j][i] for j in range(len(k_steps))]
        ax_c.plot(k_steps, ci, 'o-', color='steelblue', lw=2, ms=4, label='P(Kl.1)')
        ax_c.axhline(0.5, color='k', ls='--', lw=1); ax_c.axvline(0.5, color='cyan', ls=':', lw=1)
        ax_c.set_xlim(-0.05, 1.05); ax_c.set_ylim(-0.05, 1.1)
        ax_c.set_xlabel('k', fontsize=7); ax_c.set_ylabel('Konfidenz', fontsize=7)
        ax_c.tick_params(labelsize=6); ax_c.legend(fontsize=6, loc='upper left')
        fc = 'limegreen' if flipped else 'tomato'
        ax_c.set_title(f'{CLASS_NAMES[s_lbl]} → {CLASS_NAMES[c_lbl]}\n'
                       f'{"✓ Gekippt" if flipped else "✗ Nicht gekippt"}', fontsize=7, color=fc)

    plt.suptitle('Mirror-CFE — KFE-Übergang (k=0 bis k=1), dekodiert mit Mirror-Decoder\n'
                 'Cyan = Grenze (k=0.5) | Grün/Rot = Reflexion (k=1)', fontsize=11, y=1.01)
    plt.savefig(save_path, dpi=120, bbox_inches='tight'); plt.show()
    print(f'Gespeichert → {save_path}')

print('Inferenz- & Visualisierungsfunktionen definiert ✓')


In [ ]:
sample_images, sample_labels, sample_fnames = next(iter(test_loader))
visualise_mirror_cfe(classifier, decoder, sample_images, sample_labels,
                     n_samples=4, num_iterations=LBFGS_ITERS,
                     save_path=os.path.join(OUT_DIR, 'mirror_cfe_mnist_explanation.png'))


## 14. Sanity Check — Flip Rate (Validity im Feature-Raum)
Misst, wie oft der reflektierte/verfeinerte Feature-Vektor laut Kopf zur Zielklasse kippt.

In [ ]:
mirror_fv_chk, *_rest, cfe_labels_chk, source_labels_chk, _ = compute_mirror_cfe(
    classifier, sample_images, DEVICE, num_iterations=LBFGS_ITERS)
cfe_probs_chk = predict_from_features(classifier, mirror_fv_chk).cpu()
cfe_preds_chk = (cfe_probs_chk >= 0.5).long()
flip_rate     = (cfe_preds_chk == cfe_labels_chk.cpu()).float().mean()

print(f'Batch-Größe   : {len(sample_labels)}')
print(f'Flip Rate     : {flip_rate:.2%}  (Ziel: >80%)')
print(f'Quellvorhers. : {source_labels_chk.tolist()}')
print(f'CFE-Vorhers.  : {cfe_preds_chk.tolist()}')
print(f'Zielklassen   : {cfe_labels_chk.tolist()}')
if flip_rate < 0.8:
    print('\n⚠ Flip Rate niedrig — LBFGS_ITERS erhöhen (z.B. 50)')


## 15. Metriken
Proximity (L1), Sparsity Rate, LPIPS, FID, Validity, Denoised Validity, Coverage,
Efficiency — über die mit dem Mirror-Decoder dekodierten CFE-Bilder (k=1, verfeinert).
EBPG entfällt (keine Bounding-Boxen für MNIST). LPIPS/FID duplizieren den Graukanal auf 3.

In [ ]:
pip install lpips


In [ ]:
from scipy import linalg as scipy_linalg

try:
    import lpips
    lpips_fn = lpips.LPIPS(net='squeeze').to(DEVICE); lpips_fn.eval()
    LPIPS_AVAILABLE = True
    print('LPIPS geladen ✓')
except ImportError:
    LPIPS_AVAILABLE = False
    print('⚠ lpips nicht verfügbar — pip install lpips')


def compute_mirror_cfe_full(model, decoder, images, device, num_iterations=20):
    """Mirror-CFE inkl. Dekodierung über den Mirror-Decoder (k=1, verfeinert)."""
    mirror_fv, f1, f2, f3, cfe_labels, source_labels, orig_probs = compute_mirror_cfe(
        model, images, device, num_iterations)
    with torch.no_grad():
        cfe_image = decoder(mirror_fv.detach(), f1, f2, f3, Wmat, 1.0, source_labels, cfe_labels)
        cfe_probs = predict_from_features(model, mirror_fv.detach())
    return cfe_image, cfe_labels, source_labels, orig_probs, cfe_probs


def compute_l1(orig_np, cfe_np):
    diffs = np.abs(orig_np - cfe_np)
    return float((diffs.sum(axis=(1,2,3)) / np.prod(orig_np.shape[1:])).mean())

def compute_sparsity_rate(orig_np, cfe_np, threshold=1e-4):
    diff = np.abs(orig_np - cfe_np).mean(axis=-1)
    return float((diff > threshold).astype(float).mean(axis=(1,2)).mean())

def _to_3ch(a):
    return np.repeat(a, 3, axis=-1) if a.shape[-1] == 1 else a

def compute_lpips(orig_np, cfe_np):
    if not LPIPS_AVAILABLE:
        return None
    o = torch.tensor(_to_3ch(orig_np), dtype=torch.float32).permute(0,3,1,2).to(DEVICE) * 2 - 1
    c = torch.tensor(_to_3ch(cfe_np),  dtype=torch.float32).permute(0,3,1,2).to(DEVICE) * 2 - 1
    with torch.no_grad():
        return float(lpips_fn(o, c).mean().cpu())

from torchvision.models import inception_v3

@torch.no_grad()
def extract_inception_features(imgs_np, batch_size=32):
    imgs_np = _to_3ch(imgs_np)
    if not hasattr(extract_inception_features, '_model'):
        m = inception_v3(weights='DEFAULT', transform_input=False); m.fc = nn.Identity()
        extract_inception_features._model = m.eval().to(DEVICE)
    inc = extract_inception_features._model
    feats = []
    for i in range(0, len(imgs_np), batch_size):
        t = torch.tensor(imgs_np[i:i+batch_size], dtype=torch.float32).permute(0,3,1,2).to(DEVICE)
        t = F.interpolate(t, size=(299,299), mode='bilinear', align_corners=False)
        feats.append(inc(t).cpu().numpy())
    return np.concatenate(feats, axis=0)

def compute_fid(real_np, fake_np):
    rf, ff = extract_inception_features(real_np), extract_inception_features(fake_np)
    diff = rf.mean(0) - ff.mean(0)
    sig_r, sig_f = np.cov(rf, rowvar=False), np.cov(ff, rowvar=False)
    covmean, _ = scipy_linalg.sqrtm(sig_r @ sig_f, disp=False)
    if np.iscomplexobj(covmean): covmean = covmean.real
    return float(diff @ diff + np.trace(sig_r + sig_f - 2 * covmean))

def compute_validity(model, cfe_imgs_tensor, cfe_labels, denoise_sigma=None):
    from torchvision.transforms.functional import gaussian_blur
    model.eval(); cfe_imgs_tensor = cfe_imgs_tensor.to(DEVICE)
    if denoise_sigma is not None:
        ksz = max(int(denoise_sigma * 6) | 1, 3)
        cfe_imgs_tensor = gaussian_blur(cfe_imgs_tensor, kernel_size=[ksz, ksz],
                                        sigma=[denoise_sigma, denoise_sigma])
    with torch.no_grad():
        preds = model((cfe_imgs_tensor - 0.5) / 0.5).argmax(dim=1).cpu()
    return float((preds == cfe_labels.cpu()).float().mean())

def compute_coverage(model, decoder, images, n_runs=5, num_iterations=20):
    """Mirror-CFE ist deterministisch (L-BFGS aus geometrischer Init), std ~ 0 — erwartet."""
    images = images.to(DEVICE); valid = []
    for run in range(n_runs):
        torch.manual_seed(run)
        _, c_lbls, _, _, cfe_p = compute_mirror_cfe_full(model, decoder, images, DEVICE, num_iterations)
        valid.append(((cfe_p >= 0.5).long().cpu() == c_lbls.cpu()).float().mean().item())
    return float(np.mean(valid)), float(np.std(valid))

print('Metrik-Funktionen definiert ✓')


## 16. Fester Eval-Satz
Ein fester, reproduzierbarer Subset des Test-Sets.

In [ ]:
EVAL_SEED = 42
N_EVAL_IMAGES = min(200, len(test_ds))

_rng = np.random.RandomState(EVAL_SEED)
_all_test_idx = np.arange(len(test_ds)); _rng.shuffle(_all_test_idx)
eval_indices = _all_test_idx[:N_EVAL_IMAGES].tolist()

eval_dataset = torch.utils.data.Subset(test_ds, eval_indices)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
print(f'Eval-Satz: {len(eval_dataset)} Bilder (Seed {EVAL_SEED})')


## 17. Metriken berechnen

In [ ]:
DENOISE_SIGMA = 1.0

all_l1, all_sparsity, all_lpips = [], [], []
all_real_np, all_cfe_np = [], []
all_eval_imgs, all_cfe_tensor, all_cfe_labels = [], [], []
total_time, n_images = 0.0, 0

print(f'Berechne Metriken über {len(eval_dataset)} Bilder (fester Eval-Satz)...')
print('-' * 60)

for batch_imgs, batch_lbls, batch_fnames in tqdm(eval_loader, desc='Metrik-Evaluation'):
    t0 = time.time()
    b_cfe_img, b_cfe_lbl, b_src, _, b_cfe_p = compute_mirror_cfe_full(
        classifier, decoder, batch_imgs, DEVICE, num_iterations=LBFGS_ITERS)
    total_time += time.time() - t0
    n_images   += len(batch_lbls)
    all_eval_imgs.append(batch_imgs.cpu())

    orig_np = denormalise(batch_imgs.cpu()).permute(0, 2, 3, 1).numpy()
    cfe_np  = tanh_to_img(b_cfe_img.detach().cpu()).permute(0, 2, 3, 1).numpy().clip(0, 1)

    all_l1.append(compute_l1(orig_np, cfe_np))
    all_sparsity.append(compute_sparsity_rate(orig_np, cfe_np))
    if LPIPS_AVAILABLE:
        all_lpips.append(compute_lpips(orig_np, cfe_np))

    all_real_np.append(orig_np); all_cfe_np.append(cfe_np)
    all_cfe_tensor.append(torch.tensor(cfe_np, dtype=torch.float32).permute(0, 3, 1, 2))
    all_cfe_labels.append(b_cfe_lbl.cpu())

fid_score = compute_fid(np.concatenate(all_real_np, 0), np.concatenate(all_cfe_np, 0))

all_cfe_tensor_cat = torch.cat(all_cfe_tensor, dim=0)
all_cfe_labels_cat = torch.cat(all_cfe_labels, dim=0)
validity          = compute_validity(classifier, all_cfe_tensor_cat, all_cfe_labels_cat)
denoised_validity = compute_validity(classifier, all_cfe_tensor_cat, all_cfe_labels_cat,
                                     denoise_sigma=DENOISE_SIGMA)

print('Berechne Coverage (5 Runs über den festen Eval-Satz)...')
coverage_mean, coverage_std = compute_coverage(
    classifier, decoder, torch.cat(all_eval_imgs, dim=0), n_runs=5, num_iterations=LBFGS_ITERS)

efficiency = total_time / n_images

print('\n' + '=' * 60)
print('METRIK-ERGEBNISSE — Mirror-CFE MNIST (7 ↔ 9), Skip-Decoder')
print('=' * 60)
print(f'\n── Proximity ───────────────────────────────────────────')
print(f'  L1-Distanz:           {np.mean(all_l1):.4f}  (↓ besser)')
print(f'\n── Interpretierbarkeit ─────────────────────────────────')
print(f'  Sparsity Rate:        {np.mean(all_sparsity):.4f}  (↓ besser)')
if LPIPS_AVAILABLE:
    print(f'  LPIPS (SqueezeNet):   {np.mean(all_lpips):.4f}  (↓ besser)')
else:
    print(f'  LPIPS:                nicht verfügbar')
print(f'\n── Plausibilität ───────────────────────────────────────')
print(f'  FID:                  {fid_score:.2f}   (↓ besser)')
print(f'\n── Funktionalität ──────────────────────────────────────')
print(f'  Validity:             {validity:.2%}  (↑ besser)')
print(f'  Denoised Validity:    {denoised_validity:.2%}  (↑ besser, σ={DENOISE_SIGMA})')
print(f'  Δ Validity:           {validity - denoised_validity:.2%}  (↓ besser = weniger adversarial)')
print(f'  Coverage:             {coverage_mean:.2%} ± {coverage_std:.2%}  (↑ besser)')
print(f'  Efficiency:           {efficiency:.4f}s / CF  (↓ besser)')
print('=' * 60)
